In [8]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [9]:
%pip install faiss-cpu langchain-community
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from langchain_core.documents import Document

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [21]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [22]:
vector_store = FAISS.from_documents(
    documents= docs,
    embedding=GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001'),
)
vector_store.save_local("my_faiss_db")

In [23]:
# Get all stored documents
docs = vector_store.docstore._dict
for id, doc in docs.items():
    print(doc.page_content)
    print(doc.metadata)

Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
{'team': 'Royal Challengers Bangalore'}
Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.
{'team': 'Mumbai Indians'}
MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.
{'team': 'Chennai Super Kings'}
Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.
{'team': 'Mumbai Indians'}
Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key 

In [24]:
import numpy as np

# Get all stored vectors/embeddings
embeddings = vector_store.index.reconstruct_n(0, vector_store.index.ntotal)
print(embeddings)

[[-0.00982054  0.02545763  0.02402782 ...  0.01414876 -0.01560954
  -0.00266117]
 [-0.01989721  0.01092714  0.01730603 ...  0.00973945 -0.0176257
  -0.00460104]
 [-0.01358705 -0.00359449  0.01163961 ...  0.01149882 -0.02098301
   0.00373549]
 [-0.01261678 -0.00227585  0.00470995 ... -0.00144414  0.00646786
  -0.00529741]
 [-0.01413488 -0.02750698  0.01302837 ...  0.00987475 -0.00950909
  -0.00272136]]


In [25]:
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='7149f1b4-fb8f-40ce-b135-e4e82043383e', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='3ea14e8f-b870-43c7-ac11-a510a99e178e', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [29]:
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='7149f1b4-fb8f-40ce-b135-e4e82043383e', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  np.float32(0.640682)),
 (Document(id='3ea14e8f-b870-43c7-ac11-a510a99e178e', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  np.float32(0.6605364))]

In [30]:
# for upadting we have to delete the doc then add new one
vector_store.delete(['d6f9588d-4ca9-44cb-bcaf-10f1c396561f'])   
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)
vector_store.add_documents([updated_doc1])

['83e66e15-a704-4d38-a5a2-ff0f871bdad8']